# 07 | Burkina Faso: Presto embeddings
**Ibrahim Kabore · WorldCereal extension · Koumbia**

In Notebook 06, I prepared 1,525 crop observations, each with 12 monthly slots. Radar coverage differs sharply by year, so I will compare two sets of embeddings generated from the exact same reference samples. The model is frozen: I do not train or fine-tune Presto here.

**Research question:** does adding available Sentinel-1 radar information improve crop classification when radar coverage varies between years?

- A: optical + meteorology + terrain, with radar masked everywhere.
- B: multisensor, with radar retained wherever available.

Both use the same August 2026 encoder, January–December context and May–October seasonal pooling. The six crop labels and split assignments from Notebook 06 remain unchanged. Use a fresh Colab **GPU** runtime (L4 is sufficient). Run in order and stop if a check fails.

[WorldCereal source](https://github.com/WorldCereal/worldcereal-classification/tree/bf8667902f583ec71fcca6b1a7bd1e48f64000f7) · [PromethEO](https://github.com/WorldCereal/prometheo) · [Presto paper](https://arxiv.org/abs/2304.14065)

## 1. Install the same software revision as the French experiment
This avoids accidentally using a different architecture or preprocessing implementation. Restart the runtime if Colab requests it after installation.

In [ ]:
import sys, subprocess
REVISION = "bf8667902f583ec71fcca6b1a7bd1e48f64000f7"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    f"worldcereal @ git+https://github.com/WorldCereal/worldcereal-classification.git@{REVISION}"])

## 2. Connect Drive and verify the runtime
We reuse the cached French model archive (if present). Burkina Faso inputs and outputs are kept in their own project directory.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, zipfile, io, requests
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from importlib.metadata import version
from prometheo.models import Presto
from prometheo.models.pooling import PoolingMethods
from prometheo.predictors import Predictors, S1_BANDS, S2_BANDS, METEO_BANDS, DEM_BANDS, NODATAVALUE
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL
from worldcereal.utils.models import load_model_artifact

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ESA_EO_Projects")
PROJECT = ROOT / "WorldCereal_Burkina_Faso"
DATA = PROJECT / "data"
OUT = PROJECT / "outputs"
MODELS = PROJECT / "models"
for folder in [OUT, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| PromethEO:", version("prometheo"),
      "| WorldCereal:", version("worldcereal"))
assert (DATA / "burkina_extractions_clean.parquet").is_file()

## 3. Load and verify the frozen encoder
The checkpoint is already trained by WorldCereal. Reusing it means applying learned weights to new satellite observations; the crop labels do not enter the encoder. Verify the exact SHA-256 fingerprint recorded in France, rather than silently substituting a newer model.

In [ ]:
EXPECTED_SHA256 = "e16dce35b55b368c1553dd6c6dfe26f39d27818f24a9d710e2922cd4f00fed88"
MODEL_URL = DEFAULT_SEASONAL_MODEL_URL
assert "202608211601.zip" in MODEL_URL, MODEL_URL
french_bundle = ROOT / "WorldCereal_Crop_Mapping/models/2026/global_seasonal_20260821.zip"
bundle = french_bundle if french_bundle.is_file() else MODELS / "global_seasonal_20260821.zip"
if not bundle.is_file():
    r = requests.get(MODEL_URL, timeout=(30,120))
    r.raise_for_status()
    assert zipfile.is_zipfile(io.BytesIO(r.content))
    bundle.write_bytes(r.content)
artifact = load_model_artifact(bundle, cache_root=Path("/content/wc_cache"), encoder_only=True)
config = artifact.manifest
encoder_path = artifact.extract_dir / config["artifacts"]["checkpoints"]["encoder_only"]
fingerprint = hashlib.sha256(encoder_path.read_bytes()).hexdigest()
assert fingerprint == EXPECTED_SHA256, f"Unexpected encoder: {fingerprint}"
declared = (config.get("backbone") or {}).get("fingerprint")
if declared:
    assert fingerprint == declared
encoder = Presto()
encoder.load_state_dict(torch.load(encoder_path, map_location="cpu", weights_only=True), strict=True)
encoder = encoder.to("cuda").eval()
encoder.requires_grad_(False)
print("Verified:", fingerprint)
print("Checkpoint:", bundle)

## 4. Read and audit the prepared monthly observations
The expected count is 1,525 field observations, not 18,300 independent examples. Verify that metadata are consistent within sample IDs and that the months belong to the observation year. Keep coordinates only for the encoder and spatial grouping, not as extra Random Forest predictors.

In [ ]:
KEYS = ["ref_id","sample_id"]
META = ["label","label_year","split","spatial_group","lon","lat","ewoc_code"]
BAND_MAP = {**{f"S2-L2A-{b}": b.replace("B0","B") for b in
                ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]},
    "S1-SIGMA0-VV":"VV", "S1-SIGMA0-VH":"VH",
    "AGERA5-PRECIP":"precipitation", "AGERA5-TMEAN":"temperature",
    "elevation":"elevation", "slope":"slope"}
BANDS = list(BAND_MAP)
FEATURES = [f"presto_ft_{i}" for i in range(128)]
POOL_MONTHS = [5,6,7,8,9,10]
BATCH = 256
raw = pd.read_parquet(DATA / "burkina_extractions_clean.parquet")
assert set(KEYS + META + BANDS + ["timestamp"]) <= set(raw.columns)
assert raw[KEYS + META].notna().all().all()
raw["timestamp"] = pd.to_datetime(raw.timestamp, utc=True).dt.tz_localize(None)
raw["month"] = raw.timestamp.dt.month
raw["label_year"] = raw.label_year.astype(int)
assert raw.timestamp.dt.year.eq(raw.label_year).all()
assert not raw.duplicated(KEYS + ["month"]).any()
assert raw.groupby(KEYS).month.nunique().eq(12).all()
assert raw.groupby(KEYS)[META].nunique().le(1).all().all()
assert np.isfinite(raw[["lon","lat"]]).all().all()
expected_split = {2017:"train",2018:"train",2020:"train",2021:"validation",2022:"test"}
assert raw.split.eq(raw.label_year.map(expected_split)).all()
print("Rows:",len(raw))
print("Samples:", len(raw[KEYS].drop_duplicates()))
display(pd.crosstab(raw.drop_duplicates(KEYS).label,raw.drop_duplicates(KEYS).split,margins=True))

## 5. Build ordered 12-month arrays
Pivot to one row per sample, with the same ordered calendar slots for every band. Raw 65535 values and nonfinite measurements represent missing data, not reflectance. Inspect value ranges before applying the same WorldCereal conversions as the France workflow. The optical-only quality screen below ensures that both experiments use the same eligible observations.

In [ ]:
raw[BANDS] = raw[BANDS].apply(pd.to_numeric, errors="coerce").replace([65535, np.inf, -np.inf],np.nan)
meta = raw.drop_duplicates(KEYS).set_index(KEYS)[META].sort_index()
wide = raw.pivot(index=KEYS,columns="month",values=BANDS).reindex(
    index=meta.index,columns=pd.MultiIndex.from_product([BANDS,range(1,13)]))
values = {b:wide[b].to_numpy(dtype=np.float32) for b in BANDS}
display(pd.DataFrame({b:pd.Series(a.ravel()).describe() for b,a in values.items()}).T)
optical = np.isfinite(values["S2-L2A-B08"])
radar = np.isfinite(values["S1-SIGMA0-VV"]) & (values["S1-SIGMA0-VV"]>0)
keep = (optical.sum(1)>=6) & (optical[:,4:10].sum(1)>=3)
print("Before common quality filter:",len(meta), "| retained:",int(keep.sum()))
meta = meta.iloc[np.flatnonzero(keep)].reset_index()
values = {b:a[keep] for b,a in values.items()}
meta["optical_months"] = optical[keep].sum(1)
meta["radar_months"] = radar[keep].sum(1)
display(pd.crosstab(meta.label,meta.split,margins=True))
assert set(meta.label)=={"Maize","Cotton","Sorghum","Soybean","Cowpea","Groundnut"}
assert (pd.crosstab(meta.label,meta.split)>0).all().all()
display(meta.groupby("label_year")[["optical_months","radar_months"]].mean().round(2))

## 6. Define the shared encoder function
The function follows the earlier verified WorldCereal input convention: encoded Sentinel-1 amplitude to dB, meteorology to metres and Kelvin, terrain unchanged. Mask B8A in both experiments. The radar switch is the only difference between A and B. Monthly temporal embeddings are averaged over May through October, yielding one 128-dimensional vector per sample. Missing inputs remain masked; no missing measurement is imputed. Save this exact function for spatial inference later.

In [ ]:
def encode(values, lon, lat, dates, use_radar):
    n,t = len(lon),len(dates)
    assert t==12
    s1 = np.full((n,1,1,t,len(S1_BANDS)),NODATAVALUE,np.float32)
    s2 = np.full((n,1,1,t,len(S2_BANDS)),NODATAVALUE,np.float32)
    meteo = np.full((n,1,1,t,len(METEO_BANDS)),NODATAVALUE,np.float32)
    dem = np.full((n,1,1,len(DEM_BANDS)),NODATAVALUE,np.float32)

    for source,band in BAND_MAP.items():
        a = np.asarray(values[source],dtype=np.float32).copy()
        assert a.shape==(n,t)
        ok = np.isfinite(a) & (a!=NODATAVALUE) & (a!=65535)
        a[~ok] = NODATAVALUE
        if band in S2_BANDS:
            if band=="B8A":
                a[:] = NODATAVALUE
            s2[:,0,0,:,S2_BANDS.index(band)] = a
        elif band in S1_BANDS:
            if use_radar:
                ok &= a>0
                a[~ok] = NODATAVALUE
                a[ok] = 20*np.log10(a[ok])-83
                s1[:,0,0,:,S1_BANDS.index(band)] = a
        elif band in METEO_BANDS:
            a[ok] /= 100000 if band=="precipitation" else 100
            meteo[:,0,0,:,METEO_BANDS.index(band)] = a
        elif band in DEM_BANDS:
            dem[:,0,0,DEM_BANDS.index(band)] = a[:,0]
        else:
            raise ValueError(band)

    dates = pd.DatetimeIndex(dates)
    stamps = np.stack([dates.day,dates.month,dates.year],axis=1)
    predictor = Predictors(s1=s1,s2=s2,meteo=meteo,dem=dem,
        latlon=np.stack([lat,lon],axis=1).astype("float32").reshape(n,1,1,2),
        timestamps=np.broadcast_to(stamps,(n,t,3)).copy())
    with torch.inference_mode():
        z = encoder(predictor,eval_pooling=PoolingMethods.TIME).reshape(n,t,128)
        keep = np.flatnonzero(dates.month.isin(POOL_MONTHS)).tolist()
        return z[:,keep,:].mean(dim=1).cpu().numpy()

## 7. Smoke test on eight samples
Verify output dimensions, finite values and reproducibility. Also test a year without radar coverage: A and B should produce the same embeddings if their radar inputs are entirely missing.

In [ ]:
def run_batch(ix, use_radar):
    yrs = meta.label_year.to_numpy()[ix]
    assert len(set(yrs))==1
    dates = pd.date_range(f"{yrs[0]}-01-01",periods=12,freq="MS")
    return encode({b:a[ix] for b,a in values.items()},
                  meta.lon.to_numpy()[ix],meta.lat.to_numpy()[ix],dates,use_radar)

ix = np.flatnonzero(meta.label_year.eq(2017))[:8]
assert len(ix)>0
a = run_batch(ix,False)
b = run_batch(ix,True)
assert a.shape==b.shape==(len(ix),128)
assert np.isfinite(a).all() and np.isfinite(b).all()
assert np.allclose(a,run_batch(ix,False),atol=1e-5)
print("Probe A:",a.shape,"| Probe B:",b.shape)
print("Mean absolute difference:",np.abs(a-b).mean())
no_radar = ~np.logical_or.reduce([np.isfinite(values[b]) & (values[b]>0)
                                  for b in ["S1-SIGMA0-VV","S1-SIGMA0-VH"]]).any(1)
ix = np.flatnonzero(meta.label_year.eq(2021) & no_radar)[:8]
if len(ix):
    assert np.allclose(run_batch(ix,False),run_batch(ix,True),atol=1e-5)
    print("Radar-free equality check passed")
print("GPU memory MB:",torch.cuda.memory_allocated()/1e6)

## 8. Generate both embedding tables
Group by calendar year to align timestamps, process batches on GPU and preserve row order. Use identical metadata and the same feature naming convention in both output files.

In [ ]:
experiments = {"optical":False,"multisensor":True}
vectors = {}
for name,use_radar in experiments.items():
    Z = np.empty((len(meta),128),dtype=np.float32)
    for year in sorted(meta.label_year.unique()):
        ids = np.flatnonzero(meta.label_year.eq(year))
        for k in tqdm(range(0,len(ids),BATCH),desc=f"{name} | {year}"):
            ix = ids[k:k+BATCH]
            Z[ix] = run_batch(ix,use_radar)
    assert np.isfinite(Z).all()
    vectors[name] = Z
    print(name,Z.shape)
assert np.allclose(vectors["optical"][no_radar],vectors["multisensor"][no_radar],atol=1e-5)
difference = np.linalg.norm(vectors["optical"]-vectors["multisensor"],axis=1)
display(pd.DataFrame({"year":meta.label_year,"L2_difference":difference}).groupby("year").L2_difference.describe().round(4))

## 9. Save embeddings and provenance
Notebook 08 will read these two tables, train supervised classifiers on the training rows and evaluate validation/test separately. The spatial groups are also preserved. The 2022 test set must not be used to choose feature configuration or hyperparameters.

In [ ]:
for name,use_radar in experiments.items():
    result = pd.concat([meta,pd.DataFrame(vectors[name],columns=FEATURES)],axis=1)
    assert not result.duplicated(KEYS).any()
    path = OUT / f"reference_embeddings_{name}.parquet"
    result.to_parquet(path,index=False)
    manifest = {
        "checkpoint_sha256":fingerprint,"model_url":MODEL_URL,
        "worldcereal_revision":"bf8667902f583ec71fcca6b1a7bd1e48f64000f7",
        "worldcereal":version("worldcereal"),"prometheo":version("prometheo"),
        "context":"Jan-Dec","pool_months":POOL_MONTHS,"mask_b8a":True,
        "use_radar":use_radar,"features":FEATURES,"sample_count":len(meta),
        "split_counts":meta.split.value_counts().to_dict(),
        "quality_filter":"B08 >=6 annual months and >=3 May-Oct months",
        "spatial_block_m":2000
    }
    (MODELS / f"embedding_manifest_{name}.json").write_text(json.dumps(manifest,indent=2))
    print("Saved:",path,"|",result.shape)
print("Both configurations ready for Notebook 08.")

## 10. Interpretation and next step
An embedding is a learned numerical representation, not a crop prediction. The experiment tests whether the two input configurations yield different downstream classifier scores on the same samples. Missing radar in 2020 and 2021 is a property of the downloaded extractions; it must not be interpreted as satellite absence. Any pretraining overlap with JECAM is unknown, and temporal holdout does not imply spatial independence.

Next: Notebook 08 will compare Random Forest and histogram gradient boosting, with macro F1, balanced accuracy, confusion matrices and spatially grouped validation.